# Session 2: From Static Weights to Adaptive Allocation

In Session 1, we built a classical minimum-variance portfolio and watched it fail under stress. The core problem? Static weights can't respond to a changing world. In this session, we take humans _partially_ out of the loop: we design an AI rebalancing engine that uses **utility maximization** to allocate capital, reads market sentiment to adjust risk preferences dynamically, and enforces explicit safety rules.

> __Learning Objectives:__
>
> * __Cobb-Douglas Utility Maximization:__ Formulate portfolio allocation as a budget-constrained Cobb-Douglas utility problem and derive the closed-form analytical solution. Compare the allocation behavior of Cobb-Douglas to CES and log-linear alternatives under the same preference weights.
> * __Sentiment-Driven Preference Weights:__ Construct a sentiment signal from EMA crossover and map it to a risk-aversion parameter that modulates preference weights. Feed the signal into SIM-based preference weights so the allocator adapts dynamically to changing market conditions.
> * __Rebalancing Engine with Safety Rules:__ Implement a daily rebalancing loop that computes allocations via utility maximization and enforces trigger rules. Configure drawdown limits, turnover caps, and reallocation schedules that encode human safety judgment into the engine.

Let's build a machine that adapts.

___

## Examples
The following example notebooks accompany this lecture. They contain executable code that implements the concepts discussed here:

> __Session 2 Examples:__
>
> * [▶ Building the Cobb-Douglas Allocator](eCornell-AI-Finance-S2-Example-BuildCobbDouglasAllocator-May-2026.ipynb). We generate a synthetic market path, compute SIM parameters and the sentiment signal, build the Cobb-Douglas utility allocator, and compare its allocation behavior to CES and log-linear alternatives.
> * [▶ Rebalancing Engine and Scorecard](eCornell-AI-Finance-S2-Example-RebalancingEngineScorecard-May-2026.ipynb). We wire the Cobb-Douglas allocator into a full rebalancing engine with trigger rules, produce a scorecard comparing strategies, and run a CES elasticity sweep.
> * [▶ Static vs Adaptive Comparison](eCornell-AI-Finance-S2-Example-StaticVsAdaptiveComparison-May-2026.ipynb). We run static and adaptive strategies head-to-head on 5,000 synthetic paths, measure win rates, identify regime dependence, and compare paired quantiles across the wealth distribution.
> * [▶ Regime-Aware Sentiment](eCornell-AI-Finance-S2-Example-RegimeAwareSentiment-May-2026.ipynb). We replace EMA crossover with the HMM regime posterior as the lambda signal and compare distributional performance across 5,000 futures.
> * [▶ Stress-Testing the Rebalancing Engine](eCornell-AI-Finance-S2-Example-StressTestRebalancingEngine-May-2026.ipynb). We run all strategies on the full Monte Carlo ensemble and compute the tail-risk scorecard (VaR, CVaR, drawdown, NPV).
> * [▶ Turnover Cost, Attribution, and Diagnostics](eCornell-AI-Finance-S2-Example-TurnoverAttributionDiagnostics-May-2026.ipynb). We measure transaction cost sensitivity, decompose performance via 3-way attribution, and visualize the rebalance event timeline.

___

## Why Static Weights Fail

Session 1 showed us that the minimum-variance optimizer treats its inputs, the expected growth rate vector $\mathbb{E}[\mathbf{g}]$ and the covariance matrix $\boldsymbol{\Sigma}_g$, as fixed constants. But markets are not stationary. Correlations spike during crises, volatilities cluster, and expected returns shift with economic regimes. A portfolio computed from last year's data has no mechanism to react when the world changes _today_.

> __The core limitation:__
>
> Static portfolio construction is a _one-shot_ decision: estimate, optimize, deploy. There is no feedback loop. No matter what happens after deployment, the weights don't move until a human intervenes. This is not a flaw in the math; it is a flaw in the _architecture_.

The fix is straightforward in concept: replace the one-shot decision with a _loop_ that continuously reads market conditions and adjusts the portfolio accordingly. But we need a principled framework for making those allocation decisions. That framework is **utility maximization**.

## Utility Maximization: A Framework for Allocation

Instead of asking "what portfolio minimizes variance?" we ask a different question: **given a budget and current market conditions, what portfolio maximizes the investor's utility?**

This shift is fundamental. Minimum-variance optimization treats all assets symmetrically; the only inputs are returns and covariances. Utility maximization lets us encode _preferences_: which assets do we like right now? How strongly? And those preferences can change every day based on market signals.

> __The general problem:__
>
> Given $K$ assets with current prices $P_1, \ldots, P_K$ and a total budget $B$, choose the number of shares $n_1, \ldots, n_K$ to:
>
> $$\max_{n_1, \ldots, n_K} \quad U(n_1, \ldots, n_K) \qquad \text{subject to} \quad \sum_{i=1}^{K} n_i \cdot P_i = B$$
>
> The utility function $U$ encodes the investor's preferences. Different choices of $U$ lead to different allocation strategies.

The key insight is that the preference weights $\gamma_i$ inside $U$ are not fixed; they are computed _dynamically_ from market sentiment and fundamentals. This is where the AI enters: it translates real-time signals into preference weights that drive the allocation.

## Cobb-Douglas Utility

The **Cobb-Douglas utility function** is the workhorse of this course. It has a clean analytical solution, encodes preferences naturally through exponents, and separates assets into "preferred" and "non-preferred" categories.

> __Cobb-Douglas Utility:__
>
> $$U(n_1, \ldots, n_K) = \kappa(\boldsymbol{\gamma}) \prod_{i=1}^{K} n_i^{\gamma_i}$$
>
> where $\gamma_i \in (-1, 1)$ is the preference exponent for asset $i$, and $\kappa = +1$ if all $\gamma_i \geq 0$, else $\kappa = -1$. The preference exponents do all the work:
> * $\gamma_i > 0$, **preferred**: the investor wants more of this asset. Higher $\gamma_i$ means stronger preference.
> * $\gamma_i \leq 0$, **non-preferred**: the investor wants to minimize exposure. These assets receive only a minimum position $\epsilon$.
>
> The Cobb-Douglas utility is a product of powers, which means the optimal allocation will be proportional to the preference weights raised to those powers. The budget constraint ensures we can't just buy infinite shares of the preferred assets.

The budget-constrained Cobb-Douglas problem has a closed-form solution (no numerical optimizer needed). 

> __Optimal allocation:__
>
> Partition assets into preferred $S^+ = \{i : \gamma_i > 0\}$ and non-preferred $S^- = \{i : \gamma_i \leq 0\}$. Then:
> $$n_i^{\star} = \frac{\gamma_i}{\sum_{j \in S^+} \gamma_j} \cdot \frac{B_{\text{adj}}}{P_i} \qquad \text{for } i \in S^+$$
>
> $$n_i^{\star} = \epsilon \qquad \text{for } i \in S^-$$
>
> where $B_{\text{adj}} = B - \epsilon \sum_{k \in S^-} P_k$ is the budget remaining after funding minimum positions.

This is the core result: the budget is split among preferred assets in proportion to their preference weights, adjusted for price. The allocation responds immediately to changes in $\gamma_i$; no matrix inversion, no quadratic program, no numerical solver.

> __Example__
>
> [▶ Let's build the Cobb-Douglas allocator and compare utility functions](eCornell-AI-Finance-S2-Example-BuildCobbDouglasAllocator-May-2026.ipynb). We generate a synthetic market path, compute SIM parameters and the sentiment signal, build the Cobb-Douglas utility allocator, and compare its allocation behavior to CES and log-linear alternatives.

## From Sentiment to Preferences: The SIM-Based $\gamma_i$

The Cobb-Douglas allocator needs preference weights $\gamma_i$ as input. Where do they come from? From a combination of **fundamental signals** (the Single Index Model) and a **sentiment signal** $\lambda_t$ that we define in the next section. Each asset $i$ has **Single Index Model (SIM) parameters** $(\alpha_i, \beta_i, \sigma_i)$ estimated from historical data:
* $\alpha_i$: firm-specific excess return (Jensen's alpha)
* $\beta_i$: sensitivity to the market factor (systematic risk)
* $\sigma_i$: residual volatility

The preference weight $\gamma_i$ combines SIM fundamentals with the sentiment signal $\lambda_t$:

> __Preference Weight Computation:__
>
> $$\gamma_i = \tanh\left(\frac{\alpha_i}{\beta_i^{\lambda_t}} + \frac{\beta_i}{\beta_i^{\lambda_t}} \cdot \mathbb{E}[g_m]\right)$$
>
> which simplifies to:
>
> $$\gamma_i = \tanh\left(\frac{\alpha_i}{\beta_i^{\lambda_t}} + \beta_i^{1-\lambda_t} \cdot \mathbb{E}[g_m]\right)$$

The risk factor $\beta_i^{\lambda_t}$ is the key mechanism:
* **When bearish** ($\lambda_t > 0$): High-beta assets are penalized exponentially; the denominator grows, pushing $\gamma_i$ toward zero or negative.
* **When bullish** ($\lambda_t < 0$): High-beta assets are amplified; the risk factor shrinks, letting the growth signal dominate.
* **When neutral** ($\lambda_t = 0$): $\beta_i^0 = 1$, so the raw SIM growth score drives preferences.

The $\tanh$ activation maps the result to $(-1, 1)$, giving us well-behaved Cobb-Douglas exponents. These preference weights then flow directly into the Cobb-Douglas analytical solution to determine share allocations.

> __Example__
>
> [▶ Let's replace EMA crossover with the HMM regime posterior](eCornell-AI-Finance-S2-Example-RegimeAwareSentiment-May-2026.ipynb). We build a regime-aware lambda signal from the HMM posterior, run the engine with both EMA and regime-based sentiment on the same path, and compare their distributional performance across 5,000 synthetic futures.

## The Sentiment Signal: EMA Crossover → $\lambda$

The preference weight formula in the previous section depends on a sentiment signal $\lambda_t$ that acts as a risk-aversion dial. When $\lambda_t$ is large, the allocator penalizes high-beta assets; when small, it lets growth signals dominate. We compute $\lambda_t$ from the crossover of two exponential moving averages.

> __Exponential Moving Average:__
>
> The EMA of a price series $S_t$ with window $N$ is:
>
> $$\bar{S}_{t} = \alpha\, S_{t} + (1 - \alpha)\,\bar{S}_{t-1}, \qquad \alpha = \frac{2}{N + 1}$$
>
> We compute two EMAs: a **short-term** ($N_0 = 21$ days, ~1 month) that tracks recent momentum, and a **long-term** ($N_1 = 63$ days, ~1 quarter) that tracks the underlying trend.

The crossover between short and long EMAs produces the sentiment signal:

> __The Lambda Signal:__
>
> $$\lambda_t = -G \cdot \left(\frac{\bar{S}^{\text{short}}_t}{\bar{S}^{\text{long}}_t} - 1\right)$$
>
> where $G > 0$ is a gain constant that controls sensitivity. The interpretation:
> * $\lambda_t > 0$ → **Bearish** (short EMA below long EMA). The engine becomes risk-averse.
> * $\lambda_t < 0$ → **Bullish** (short EMA above long EMA). The engine takes on more risk.
> * $\lambda_t \approx 0$ → **Neutral** (no strong signal).

The EMA crossover is one choice of sentiment signal. Other sources, such as NLP-derived scores from news articles, earnings calls, or regulatory filings, can also drive $\lambda_t$. The allocator framework is agnostic to the signal source; any scalar that maps market conditions to a risk-aversion level will work. We explore richer sentiment signals in Session 4.

## Alternative Utility Functions

Cobb-Douglas is not the only choice. Different utility functions encode different assumptions about how investors trade off between assets. Here we introduce two alternatives that we compare in the examples.

### CES (Constant Elasticity of Substitution)

> __CES Utility:__
>
> $$U_{\text{CES}}(\mathbf{n}) = \left(\sum_{i \in S^+} \gamma_i \cdot n_i^{\rho}\right)^{1/\rho}, \qquad \rho = \frac{\sigma - 1}{\sigma}$$
>
> where $\sigma > 0$ is the **elasticity of substitution**, controlling how willing the investor is to shift allocation between assets.

CES is a generalization of Cobb-Douglas:
* As $\sigma \to 1$: CES converges to Cobb-Douglas (unit elasticity)
* As $\sigma \to \infty$: CES converges to linear utility (perfect substitutes, all-in on the best asset)
* As $\sigma \to 0$: CES converges to Leontief (perfect complements, equal allocation)

**When to use CES:** When you want to control how concentrated the allocation becomes. High $\sigma$ produces more concentrated portfolios; low $\sigma$ produces more diversified ones.

### Log-Linear Utility

> __Log-Linear Utility:__
>
> $$U_{\text{log}}(\mathbf{n}) = \sum_{i \in S^+} \gamma_i \cdot \log(n_i)$$

Log-linear utility is the logarithmic transform of Cobb-Douglas. It produces the **same optimal allocation** (the log transform preserves the optimum), but the utility _values_ differ. This matters when utility is used as a reward signal, for example in the bandit algorithms we introduce in Session 3.

**When to use log-linear:** When you want the same allocation as Cobb-Douglas but need utility values that are additive (useful for averaging across scenarios or feeding into learning algorithms).

## The Rebalancing Engine: Pseudo-code

The utility allocator gives us a principled way to compute positions at any point in time. The rebalancing engine wraps this allocator in a daily loop with safety rules:

> __Rebalancing Engine:__
>
> **Phase 1: Initialization**
> 1. Load market price path $S_t$ for a broad index.
> 2. Compute short-term EMA ($N_0 = 21$) and long-term EMA ($N_1 = 63$). Define warmup offset $t_0 = N_0 + N_1$.
> 3. Build the sentiment series: $\lambda_t = -G \cdot (\bar{S}^{\text{short}}_t / \bar{S}^{\text{long}}_t - 1)$.
> 4. Smooth market growth into $g_{m,t}$ using an EMA of log returns ($N_2 = 10$).
> 5. Assemble the **context model**: budget $B$, ticker universe, price matrix, SIM parameters, risk floor $\epsilon$, and market factor $g_{m,t}$.
> 6. Define **trigger rules**: drawdown limit, turnover cap, and reallocation schedule.
> 7. Compute initial allocation via Cobb-Douglas utility maximization.
>
> **Phase 2: Daily Loop** (for $t = t_0 + 1, \ldots, t_0 + T$)
> 1. Read reallocation flag $a_t$ from schedule.
> 2. **If $a_t = 1$ (rebalance day):**
>    - Liquidate current positions at today's prices
>    - **Check drawdown trigger:** if drawdown exceeds $d_{\max}$, de-risk to 100% cash
>    - Otherwise, update $\lambda_t$, compute new $\gamma_i$ preferences, solve Cobb-Douglas allocation
>    - **Check turnover cap:** if proposed trades exceed $\tau_{\max}$, scale down proportionally
> 3. **If $a_t = 0$ (hold day):** Propagate prior allocation unchanged.
>
> **Phase 3: Output**
> Return history indexed by trading day: positions, preference weights, cash, and total wealth.

The following examples implement this engine and compare it against the Session 1 baselines.

> __Example__
>
> [▶ Let's wire the allocator into a rebalancing engine and produce a scorecard](eCornell-AI-Finance-S2-Example-RebalancingEngineScorecard-May-2026.ipynb). We run the Cobb-Douglas engine with trigger rules on a synthetic market path, compare it head-to-head against the Session 1 baselines, and sweep the CES elasticity parameter to see how concentration tunes engine behavior.
>
> [▶ Let's compare static and adaptive allocation across 5,000 paths](eCornell-AI-Finance-S2-Example-StaticVsAdaptiveComparison-May-2026.ipynb). We run both strategies on the full Monte Carlo ensemble, measure how often the engine wins, identify which regimes drive the edge, and compare paired quantiles across the wealth distribution.

## Trigger Rules: The Safety Net

An adaptive engine that trades freely is dangerous; it can churn the portfolio, chase noise, or fail to protect capital during a crash. Trigger rules are the _guardrails_ that keep the engine safe:

| Rule | Parameter | What It Does |
|------|-----------|-------------|
| **Drawdown Limit** | $d_{\max}$ (e.g., 10%) | If portfolio wealth drops more than $d_{\max}$ from its peak, the engine de-risks to 100% cash. This is a circuit breaker. |
| **Turnover Cap** | $\tau_{\max}$ (e.g., 50%) | If the proposed rebalance would trade more than $\tau_{\max}$ of portfolio value, the trades are scaled down proportionally. This controls transaction costs. |
| **Reallocation Schedule** | $a_t \in \{0, 1\}$ | The binary schedule determines _which days_ the engine is allowed to rebalance. Daily, weekly, or event-driven. |

> __Why are trigger rules essential?__
>
> Without them, the engine is unconstrained: it could rebalance every day (expensive), ignore a crash (catastrophic), or flip positions wildly based on noise in the lambda signal. Trigger rules encode _human judgment_ about acceptable risk into the machine's decision loop. They are the bridge between autonomous operation and investment committee oversight.

In Session 4, we extend these rules with human override protocols and escalation procedures for production deployment.

> __Example__
>
> [▶ Let's stress-test the rebalancing engine on 5,000 synthetic futures](eCornell-AI-Finance-S2-Example-StressTestRebalancingEngine-May-2026.ipynb). We generate the full Monte Carlo scenario ensemble, run all strategies, and compute the tail-risk scorecard (VaR, CVaR, drawdown, NPV) to evaluate how the engine performs under stress.
>
> [▶ Let's diagnose turnover costs and attribution](eCornell-AI-Finance-S2-Example-TurnoverAttributionDiagnostics-May-2026.ipynb). We measure transaction cost sensitivity, decompose performance into a 3-way attribution (utility choice, dynamic reallocation, trigger rules), and visualize the rebalance event timeline.

## Summary

In this session, we replaced the static minimum-variance allocation from Session 1 with an adaptive rebalancing engine that reads market conditions and adjusts the portfolio daily within explicit safety constraints.

> __Key Takeaways:__
>
> * __Utility maximization replaces variance minimization:__ The Cobb-Douglas utility function allocates budget proportionally to preference weights via a closed-form analytical solution. CES and log-linear alternatives offer different concentration and diversification trade-offs using the same preference inputs.
> * __Preferences adapt dynamically to market conditions:__ Preference weights combine SIM fundamentals with a sentiment signal derived from EMA crossover, so the allocator shifts risk appetite as conditions change. When sentiment turns bearish, high-beta assets are penalized; when bullish, they are amplified.
> * __Trigger rules encode human judgment:__ Drawdown limits, turnover caps, and reallocation schedules act as safety guardrails that constrain the engine's behavior. These rules ensure the engine operates within bounds that a human investment committee would approve.

**Next Session:** In [Session 3](../session-3/eCornell-AI-Finance-S3-Lecture-HMMBacktesting-May-2026.ipynb), we stress-test this rebalancing engine against Hidden Markov Model-generated synthetic market paths that include regime shifts. We also introduce multi-armed bandits as an adaptive challenger: can a learning algorithm beat the utility-based allocator when markets change regime?

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The rebalancing engine described here is a pedagogical tool using synthetic data and simplified models. Real-world algorithmic trading involves regulatory requirements, execution risk, and operational considerations beyond the scope of this session.